## Create list of grouped sites to do infrastructure classification

- map official IDs to grouped ID
- combine with list of sites with multiple years of data

NOTE: I first did this using agg_counts instead of just the combined counterpoint IDs, but I've changed it here to avoid a circularity problem. There are one or two differences (extra sites here) because some sites were removed at the aggregation step, but the results should be unchanged. 

In [1]:
import pandas as pd
import numpy as np

In [2]:
counterpoint_ids = pd.read_csv("data/counterpoint_id_groupings.csv")
official_count_ids_2021 = pd.read_csv("data/Official Count Locations IDs.csv")

In [3]:
# create the mapped column
official_count_ids_2021 = official_count_ids_2021.merge(counterpoint_ids[['counterpoint_id', 'grouped_counterpoint_id']], on = 'counterpoint_id')
# fill non-mapped values with the original ID
official_count_ids_2021['grouped_counterpoint_id'] = official_count_ids_2021['grouped_counterpoint_id'].fillna(official_count_ids_2021['counterpoint_id'])

In [4]:
official_sites_unique = list(official_count_ids_2021['grouped_counterpoint_id'].unique())

In [5]:
# how many are present in multiple years?
year_counts_grouped = counterpoint_ids[['grouped_counterpoint_id', 
                                        'year']].drop_duplicates().groupby(['grouped_counterpoint_id'])['year'].count().reset_index()

In [6]:
multi_year_sites_unique = list(year_counts_grouped[year_counts_grouped['year'] > 1]['grouped_counterpoint_id'].unique())

In [7]:
unique_sites_2022_onwards = counterpoint_ids[counterpoint_ids['year'] != 2021]['grouped_counterpoint_id'].unique()

### Make dataframe with counterpoint ID, counterpoint name, official site yes/no, years counted

In [8]:
df = pd.DataFrame()

df['grouped_counterpoint_id'] = list(set(official_sites_unique + multi_year_sites_unique + list(unique_sites_2022_onwards)))

In [9]:
df2 = df.merge(counterpoint_ids, on = 'grouped_counterpoint_id', how = 'left')

In [10]:
df2.shape

(861, 12)

In [11]:
counterpoint_ids.shape

(982, 12)

In [12]:
counterpoint_ids[['geocoded_city', 'geocoded_province',
       'year', 'geocoded_country', 'grouped_counterpoint_id']].drop_duplicates().shape

(828, 5)

In [13]:
df2 = df2.drop_duplicates()

In [14]:
df2.shape

(861, 12)

In [15]:
df2['official_2021_site'] = 'no'

In [16]:
df2.loc[df2['grouped_counterpoint_id'].isin(official_count_ids_2021['counterpoint_id']), 'official_2021_site'] = 'yes'

In [17]:
df2 = df2.sort_values(by = ['geocoded_province', 'geocoded_city', 'grouped_counterpoint_id'])

In [18]:
df2.to_csv("data/counterpoint_official_sites_multiyear_counts.csv", index = None)

Moreno classified using this file after the 2023 count.

### 2024 Can-BICS updates

For the 2024 data, we started from the previous classification, added any new count sites and looked for years with missing classifications. 



- "missing_canbics_with_forecasting_2021_removed" - these are all the year/site combinations that are not classified, assuming that previous high comfort sites remain high comfort, and removing unofficial sites from 2021.

- "canbics_all_filtered.csv" has all the unique year/site combinations, most of which are already classified by you. I thought it might be helpful to have all the data together to easily see if a site was previously classified.

In [19]:
canbics = pd.read_csv("data/counterpoint_official_sites_multiyear_counts-CanBICS-v2_sheet1.csv") # 2023 count inclusive

In [20]:
canbics.columns

Index(['grouped_counterpoint_id', 'cp_name', 'lat', 'lng', 'osm_id',
       'geocoded_city', 'geocoded_province', 'count_year', 'geocoded_country',
       'official_2021_site', 'can_bics_facility', 'other_cycle_facility',
       'comfort_class', 'google_street_month', 'google_street_year'],
      dtype='object')

In [21]:
canbics[['grouped_counterpoint_id', 'count_year', 'can_bics_facility', 'other_cycle_facility',
       'comfort_class', 'google_street_month', 'google_street_year']]

,grouped_counterpoint_id,count_year,can_bics_facility,other_cycle_facility,comfort_class,google_street_month,google_street_year
0,33,2022,Cycle track,NaN,High,MAY,2019.0
1,295,2021,Painted bike lane,NaN,Low,JUL,2017.0
2,295,2022,Painted bike lane,NaN,Low,JUL,2017.0
3,295,2023,Painted bike lane,NaN,Low,JUL,2017.0
4,343,2021,Non-conforming,Local street shared roadway,NaN,MAY,2019.0
...,...,...,...,...,...,...,...
502,6628,2022,Not classified,NaN,NaN,NOV,2022.0
503,6628,2023,Local street bikeway,NaN,High,JUL,2023.0
504,6632,2022,Painted bike lane,NaN,Low,MAY,2022.0
505,6632,2023,Painted bike lane,NaN,Low,JUL,2023.0


In [22]:
counterpoint_ids.shape

(982, 12)

In [23]:
counterpoint_ids_canbics = counterpoint_ids.merge(canbics[['grouped_counterpoint_id', 'count_year', 'can_bics_facility', 'other_cycle_facility',
       'comfort_class', 'google_street_month', 'google_street_year']], left_on = ['grouped_counterpoint_id', 'year'],
                      right_on = ['grouped_counterpoint_id', 'count_year'], how = 'left').drop_duplicates()

In [24]:
assert counterpoint_ids.shape[0] == counterpoint_ids_canbics.shape[0]

In [25]:
missing_canbics = counterpoint_ids_canbics[counterpoint_ids_canbics['can_bics_facility'].isna()][['year', 'grouped_counterpoint_id', 
                                                                      'grouped_counterpoint_name',
                                                                      'grouped_counterpoint_osm_id',
                                                     'grouped_counterpoint_lat', 'grouped_counterpoint_lng',
                                                                      'geocoded_city', 'geocoded_province']].drop_duplicates()

In [26]:
missing_canbics[['grouped_counterpoint_id', 'grouped_counterpoint_name']].drop_duplicates().value_counts().reset_index()

,grouped_counterpoint_id,grouped_counterpoint_name,count
0,8244.0,Hornby,1
1,32.0,WRHA (on McDermot) looking South across McDermot,1
2,295.0,Forks West Entrance At Main Street,1
3,343.0,West Broadway Bus Stop On Balmoral,1
4,766.0,Laura Secord,1
...,...,...,...
305,2248.0,University of Saskatchewan,1
306,2068.0,Esplanade Riel WTA,1
307,1931.0,Munson Park,1
308,1350.0,Yonge Finch,1


In [27]:
missing_canbics['year'].value_counts()

year
2024    167
2021    122
2023     34
2022      4
Name: count, dtype: int64

### For sites with previously-recorded high comfort infrastructure, don't need to classify again

In [28]:
high_comfort_sites = counterpoint_ids_canbics[['year', 'grouped_counterpoint_id', 'comfort_class']].sort_values(by = 'comfort_class')

In [29]:
high_comfort_sites = high_comfort_sites[high_comfort_sites['comfort_class'] == 'High']

In [30]:
# get highest recorded year for high comfort facilities
high_comfort_ids = high_comfort_sites.sort_values(by = 'year', 
                                                  ascending = False).drop_duplicates(subset = 'grouped_counterpoint_id')['grouped_counterpoint_id']

In [31]:
# set comfort class to 'High' for 2024 sites matching previous sites
counterpoint_ids_canbics.loc[(counterpoint_ids_canbics['year'] == 2024) 
    & counterpoint_ids_canbics['grouped_counterpoint_id'].isin(high_comfort_ids), 'comfort_class'] = 'High'

In [32]:
missing_with_forecast = counterpoint_ids_canbics[(counterpoint_ids_canbics['can_bics_facility'].isna()) &
          (counterpoint_ids_canbics['comfort_class'].isna()) ][['year', 'grouped_counterpoint_id', 
                                                                      'grouped_counterpoint_name',
                                                                      'grouped_counterpoint_osm_id',
                                                     'grouped_counterpoint_lat', 'grouped_counterpoint_lng',
                                                                      'geocoded_city', 'geocoded_province']].drop_duplicates()

In [33]:
missing_with_forecast.to_csv("data/missing_canbics_2024_with_forecasting.csv", index= None)

In [34]:
year_location_counts = counterpoint_ids_canbics[['year','grouped_counterpoint_id']].drop_duplicates().groupby('grouped_counterpoint_id')['year'].count().reset_index()

In [35]:
# sites counted in multiple years
multi_year_ids = list(year_location_counts[year_location_counts['year'] > 1]['grouped_counterpoint_id'])

In [36]:
missing_with_forecast[missing_with_forecast['grouped_counterpoint_id'].isin(multi_year_ids)]['year'].value_counts()

year
2024    102
2023     13
2021      4
2022      3
Name: count, dtype: int64

Send to Moreno:
- site/year combinations that are missing classification AFTER forecasting the high comfort sites to 2024
- without sites that were not counted in 2021
- UNLESS those sites were counted in multiple years

In [37]:
missing_2021_filtered = missing_with_forecast[((missing_with_forecast['grouped_counterpoint_id'].isin(multi_year_ids))
                     & (missing_with_forecast['year'] == 2021))
                     | (missing_with_forecast['year'] != 2021)].sort_values(by = ['grouped_counterpoint_id', 'year'])

In [38]:
# combine the above with previous canbics dataframe
missing_2021_filtered.columns

Index(['year', 'grouped_counterpoint_id', 'grouped_counterpoint_name',
       'grouped_counterpoint_osm_id', 'grouped_counterpoint_lat',
       'grouped_counterpoint_lng', 'geocoded_city', 'geocoded_province'],
      dtype='object')

In [39]:
# SEND THIS to Moreno - just the sites that need classifying
# rename columns to match canbics
missing_2021_filtered = missing_2021_filtered.rename(columns = {'year':'count_year', 'grouped_counterpoint_name':'cp_name',
                                        'grouped_counterpoint_osm_id':'osm_id', 'grouped_counterpoint_lat':'lat',
       'grouped_counterpoint_lng':'lng'})

missing_2021_filtered.to_csv("data/missing_canbics_with_forecasting_2021_removed.csv", index = None)

Use entire missing dataframe to include 2024 years

In [40]:
missing_canbics.loc[(missing_canbics['year'] == 2024) & missing_canbics['grouped_counterpoint_id'].isin(high_comfort_ids), 'comfort_class'] = 'High'

In [41]:
missing_canbics = missing_canbics.rename(columns = {'year':'count_year', 'grouped_counterpoint_name':'cp_name',
                                        'grouped_counterpoint_osm_id':'osm_id', 'grouped_counterpoint_lat':'lat',
       'grouped_counterpoint_lng':'lng'})

In [42]:
# remove unclassified 2021 sites unless present in multiple years 
missing_canbics_filtered = missing_canbics[((missing_canbics['grouped_counterpoint_id'].isin(multi_year_ids))
                     & (missing_canbics['count_year'] == 2021))
                     | (missing_canbics['count_year'] != 2021)]

In [43]:
missing_canbics_filtered['geocoded_country'] = 'Canada'

/tmp/ipykernel_49715/1875594731.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_canbics_filtered['geocoded_country'] = 'Canada'


In [44]:
# SEND THIS to Moreno - all sites classified and unclassified after filtering
pd.concat([missing_canbics_filtered, canbics]).sort_values(by = ['grouped_counterpoint_id', 
                                                                 'count_year']).drop(columns = ['official_2021_site',
                                                                                                'geocoded_country']).to_csv("data/canbics_all_filtered.csv", index = None)

#### Fill in missing info from years that remained high comfort

I did this by hand in the CSV "PP_CounterPoints_Cycle_Facilities_FULL_02JAN2025.csv", received from Moreno after classification. 
Procedure:
- where a site had been previously identified as High comfort, I did not ask Moreno to classify in 2024. In these cases I copied the previous year's classification.
- note: counterpoint 295 was coded as painted bike lane by Moreno but I changed it to cycle track: it has a curb separating the street. Location is Fort Gibraltar Trail in Winnipeg
- note: counterpoint 5313 (stanley park seawall) was coded as Bike path in 2021 and cycle track in 2022. At this point in the trail the bike path joins the road, making it more like a cycle track. The trail is not changed at all the dates I can see. Because the counterpoint says to coint all vehicles across the road, and there is a road there, classify it as a cycle track
- note: counterpoint 5376 was classified as a local street shared roadway in 2021 and a local street bikeway in 2022 but the infrastructure has not changed. In the October 2022 streetview image there are planters and a one-way bicycles excepted sign which is I think is why the different classification. https://www.google.ca/maps/@42.9825416,-81.2539807,3a,75y,91.52h,73.71t/data=!3m8!1e1!3m6!1sKUOoKZ1a1x3BLsgbJF1DbQ!2e0!5s20221001T000000!6shttps:%2F%2Fstreetviewpixels-pa.googleapis.com%2Fv1%2Fthumbnail%3Fcb_client%3Dmaps_sv.tactile%26w%3D900%26h%3D600%26pitch%3D16.29451843971424%26panoid%3DKUOoKZ1a1x3BLsgbJF1DbQ%26yaw%3D91.51682079857507!7i16384!8i8192?entry=ttu&g_ep=EgoyMDI1MDIwMi4wIKXMDSoASAFQAw%3D%3D.
